# negative-back — ex1: negative_back — sign flip of grad_out

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `negative-back`. Running the final beacon cell reports progress against the `Backprop: negative_back` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: negative_back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`negative-back`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "negative-back"
DD_SUBTOPIC = "Backprop: negative_back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `negative_back` — quick refresher

`negative(x) = -x` is elementwise and linear. The local derivative is constant `-1` at every position, so the chain rule collapses to a sign flip on `grad_out`.

**Worked exemplar.**
```
out  = -x                  # forward
d/dx (-x) = -1             # local derivative
grad_in = grad_out * (-1)  # chain rule
        = -grad_out
```

Shape of `grad_in` equals shape of `x` — same as `grad_out` for this op.

### Exercise 1 — negative_back — sign flip of grad_out

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Bloom level: Apply
> LO: Apply the elementwise chain rule to derive negative_back, returning grad_in = -grad_out with the same shape and dtype as x.
> Keywords: negative, elementwise, sign-flip
> ```

**KCs targeted:** `chain-rule-elementwise`, `backward-fn-signature`

Implement `negative_back(grad_out, out, x)` for the forward op `out = -x`.

Derivation:
- `d/dx (-x) = -1` at every position.
- Chain rule: `dL/dx = grad_out * (-1) = -grad_out`.

Return a `torch.Tensor` with the same shape as `x`. No autograd, no in-place mutation — return a new tensor.

In [ ]:
def negative_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    """dL/dx for out = -x."""
    raise NotImplementedError()


def _test_ex1():
    # --- scalar ---
    x = t.tensor([3.0])
    out = -x
    g = negative_back(t.tensor([1.0]), out, x)
    assert t.allclose(g, t.tensor([-1.0])), f'scalar: {g}'

    # --- vector with non-unit grad_out ---
    x = t.tensor([1.0, -2.0, 3.0])
    out = -x
    grad_out = t.tensor([5.0, 7.0, -2.0])
    g = negative_back(grad_out, out, x)
    assert g.shape == x.shape, f'shape: {g.shape}'
    assert t.allclose(g, -grad_out), f'value: {g}'

    # --- matrix shape ---
    rng = t.Generator().manual_seed(0)
    X = t.randn(3, 4, generator=rng)
    G = t.randn(3, 4, generator=rng)
    g = negative_back(G, -X, X)
    assert g.shape == (3, 4)
    assert t.allclose(g, -G)

    # --- not in-place: grad_out must be unchanged ---
    g_in = t.tensor([1.0, 2.0, 3.0])
    g_in_copy = g_in.clone()
    _ = negative_back(g_in, -t.tensor([0.0, 0.0, 0.0]), t.tensor([0.0, 0.0, 0.0]))
    assert t.allclose(g_in, g_in_copy), 'negative_back must not mutate grad_out'

    # --- witness vs torch.autograd ---
    x_ref = t.tensor([1.5, -0.3, 2.7], requires_grad=True)
    y = (-x_ref).sum()
    y.backward()
    x_det = x_ref.detach()
    g_ours = negative_back(t.ones(3), -x_det, x_det)
    assert t.allclose(g_ours, x_ref.grad, atol=1e-6), (
        f'disagrees with autograd: ours={g_ours}, ref={x_ref.grad}'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def negative_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d/dx (-x) = -1; chain rule collapses to grad_in = -grad_out.
    return -grad_out
```

**One-liner.** Elementwise + constant derivative = pure sign flip.

**Why neither `out` nor `x` is read.** The local derivative is the constant `-1` — independent of position and of the input value. Some back fns ignore `out`, some ignore `x`, some use both. The uniform signature carries all three so any back fn can be dispatched the same way.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()